Clip points for QGIS

In [7]:
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib import rcParams
import matplotlib.patheffects as path_effects
from matplotlib_scalebar.scalebar import ScaleBar

from shapely.geometry import box, LineString, Point, MultiPoint
from shapely.ops import unary_union, polygonize

from scipy.optimize import curve_fit
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, r2_score, mean_squared_error

import hdbscan
from collections import Counter

projected_crs = "EPSG:21037"
DATA_DIR = Path('/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/')

In [8]:
boundary = gpd.read_file("/Users/wenlanzhang/Downloads/PhD_UCL/Data/Shp/NariobiShp/Shp_from_Constituency/Nairobi_shp_C.shp")
boundary.set_crs('EPSG:4326', allow_override=True, inplace=True)
boundary

,country,provpcode,province,ctypcode,county,scpcode,subcounty,dhis2_id,geometry
0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi,"MULTIPOLYGON (((36.92032 -1.3518, 36.92037 -1...."


In [4]:
AllImage_df = pd.read_csv(DATA_DIR / "img/Combined_SVI.csv")
# AllImage_df
GSVI = AllImage_df[(AllImage_df['img_dir'] != 'ZWL/') & (AllImage_df['img_dir'] != 'Faith/')]
GSVI_geometry = [Point(lon, lat) for lon, lat in zip(GSVI['lon'], GSVI['lat'])]
GSVI_gdf = gpd.GeoDataFrame(GSVI, geometry=GSVI_geometry)
GSVI_gdf.set_crs('EPSG:4326', allow_override=True, inplace=True)
GSVI_Nairobi = gpd.sjoin(GSVI_gdf, boundary, how='inner', predicate='within') 
GSVI_Nairobi

/var/folders/2_/nk9j6sb901n_fk5dz_9vtqj80000gn/T/ipykernel_3731/3012455576.py:1: DtypeWarning: Columns (14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  AllImage_df = pd.read_csv(DATA_DIR / "img/Combined_SVI.csv")


,img_name,year,month,day,hour,lat,lon,panoid,img_dir,exist,...,geometry,index_right,country,provpcode,province,ctypcode,county,scpcode,subcounty,dhis2_id
817,KdbXvhlAf5Sqjw-hwBb_Ww_0,2022,2,NaN,NaN,-1.240368,36.928039,KdbXvhlAf5Sqjw-hwBb_Ww,Google/K/d/,True,...,POINT (36.92804 -1.24037),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi
818,1Xyfr0j4E3Gx6_Coadexow_180,2018,2,NaN,NaN,-1.282457,36.751317,1Xyfr0j4E3Gx6_Coadexow,Google/1/X/,True,...,POINT (36.75132 -1.28246),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi
819,AgROMWzI6TjbZRpV6iKJDg_270,2021,7,NaN,NaN,-1.284253,36.902957,AgROMWzI6TjbZRpV6iKJDg,Google/A/g/,True,...,POINT (36.90296 -1.28425),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi
820,AgROMWzI6TjbZRpV6iKJDg_180,2021,7,NaN,NaN,-1.284253,36.902957,AgROMWzI6TjbZRpV6iKJDg,Google/A/g/,True,...,POINT (36.90296 -1.28425),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi
821,nTRlyH7aG8CmTfOI1_akgw_0,2018,3,NaN,NaN,-1.279155,36.719541,nTRlyH7aG8CmTfOI1_akgw,Google/n/T/,True,...,POINT (36.71954 -1.27915),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
454170,JFXDvor92W2kYc3_-q8h3A_180,2021,7,NaN,NaN,-1.317753,36.716709,JFXDvor92W2kYc3_-q8h3A,Google/J/F/,True,...,POINT (36.71671 -1.31775),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi
454171,al7xV7LBhDBo5aFQAj5esQ_0,2018,3,NaN,NaN,-1.282883,36.725289,al7xV7LBhDBo5aFQAj5esQ,Google/a/l/,True,...,POINT (36.72529 -1.28288),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi
454172,al7xV7LBhDBo5aFQAj5esQ_90,2018,3,NaN,NaN,-1.282883,36.725289,al7xV7LBhDBo5aFQAj5esQ,Google/a/l/,True,...,POINT (36.72529 -1.28288),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi
454173,al7xV7LBhDBo5aFQAj5esQ_270,2018,3,NaN,NaN,-1.282883,36.725289,al7xV7LBhDBo5aFQAj5esQ,Google/a/l/,True,...,POINT (36.72529 -1.28288),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi


In [5]:
GSVI_Nairobi.to_csv('/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/QGIS/GSVI_Nairobi.csv', index=False) 

In [6]:
AllWaste_df = pd.read_csv(DATA_DIR / "img/Correct_SVI.csv")
AllWaste_df = AllWaste_df[(AllWaste_df['img_dir'] != 'ZWL/') & (AllWaste_df['img_dir'] != 'Faith/')]
AllWaste_df_geometry = [Point(lon, lat) for lon, lat in zip(AllWaste_df['lon'], AllWaste_df['lat'])]
AllWaste_gdf = gpd.GeoDataFrame(AllWaste_df, geometry=AllWaste_df_geometry)
AllWaste_gdf.set_crs('EPSG:4326', allow_override=True, inplace=True)
Waste_Nairobi = gpd.sjoin(AllWaste_gdf, boundary, how='inner', predicate='within')
Waste_Nairobi

,img_name,year,month,day,hour,lat,lon,panoid,img_dir,exist,...,geometry,index_right,country,provpcode,province,ctypcode,county,scpcode,subcounty,dhis2_id
73,1Xyfr0j4E3Gx6_Coadexow_180,2018,2,NaN,NaN,-1.282457,36.751317,1Xyfr0j4E3Gx6_Coadexow,Google/1/X/,True,...,POINT (36.75132 -1.28246),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi
74,nTRlyH7aG8CmTfOI1_akgw_0,2018,3,NaN,NaN,-1.279155,36.719541,nTRlyH7aG8CmTfOI1_akgw,Google/n/T/,True,...,POINT (36.71954 -1.27915),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi
75,uc2yRcVOf6bCSyBKfnm0Sw_90,2018,2,NaN,NaN,-1.283062,36.751316,uc2yRcVOf6bCSyBKfnm0Sw,Google/u/c/,True,...,POINT (36.75132 -1.28306),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi
76,y_-_BCz3RPfZPFkLoqlA7Q_0,2018,2,NaN,NaN,-1.285056,36.745694,y_-_BCz3RPfZPFkLoqlA7Q,Google/y/_/,True,...,POINT (36.74569 -1.28506),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi
77,y_-_BCz3RPfZPFkLoqlA7Q_90,2018,2,NaN,NaN,-1.285056,36.745694,y_-_BCz3RPfZPFkLoqlA7Q,Google/y/_/,True,...,POINT (36.74569 -1.28506),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3965,K2bDOcQT-fC73J3sTszcJw_0,2021,8,NaN,NaN,-1.314196,36.889760,K2bDOcQT-fC73J3sTszcJw,Google/K/2/,True,...,POINT (36.88976 -1.3142),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi
3966,A2MMbVUET-qhK8hNcteVBw_180,2018,2,NaN,NaN,-1.276726,36.919951,A2MMbVUET-qhK8hNcteVBw,Google/A/2/,True,...,POINT (36.91995 -1.27673),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi
3967,zVDYat_guuNnsZ9DemVKwA_180,2018,2,NaN,NaN,-1.276957,36.919431,zVDYat_guuNnsZ9DemVKwA,Google/z/V/,True,...,POINT (36.91943 -1.27696),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi
3968,u_Zx9slVChF217zkkuGJnw_0,2021,8,NaN,NaN,-1.313337,36.872663,u_Zx9slVChF217zkkuGJnw,Google/u/_/,True,...,POINT (36.87266 -1.31334),0,Kenya,KEN_1_8,Nairobi,KEN_2_47,Nairobi,KEN_3_279,Roysambu,j7GpbairCOi


In [7]:
Waste_Nairobi.to_csv('/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/QGIS/Waste_Nairobi.csv', index=False) 

In [9]:
slum = gpd.read_file("/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/Angela/slumaps_nairobi_sett/slumaps_nairobi_sett.shp")
slum.set_crs('EPSG:4326', allow_override=True, inplace=True)
slum_dissolved = slum.dissolve()
slum_dissolved = slum_dissolved[['id', 'geometry']]
slum_dissolved = gpd.clip(slum_dissolved, boundary)

slum_dissolved

,id,geometry
0,719.0,"MULTIPOLYGON (((36.73436 -1.30649, 36.73436 -1..."


In [10]:
slum_dissolved.to_file("/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/QGIS/slum_dissolved.geojson", driver='GeoJSON')